# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

**Formulation de vos hypothèses  :**
- **Test #1 — Mann-Whitney :** H₀ = les distributions des retards à l’arrivée sont identiques pour les deux aéroports comparés ; H₁ = elles diffèrent. Ce test non paramétrique convient à la comparaison de deux groupes indépendants lorsque les données peuvent être asymétriques.
- **Test #2 — Spearman :** H₀ = il n’existe pas d’association monotone entre la distance du vol et son retard à l’arrivée ; H₁ = il en existe une. Cette corrélation de rangs ne suppose pas une distribution normale.

In [3]:
SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
silver_df = spark.read.format("delta").load(SILVER_MAIN_PATH)

# Comparer les retards des deux aéroports avec le plus de vols.
top_airports = [
    row["origin_airport"]
    for row in (
        silver_df
        .groupBy("origin_airport")
        .count()
        .orderBy(F.desc("count"), F.asc("origin_airport"))
        .limit(2)
        .collect()
    )
]
assert len(top_airports) == 2, "Il faut au moins deux aéroports dans la table Silver."

# Échantillons aléatoires limités pour ne pas rapatrier toute la table.
groups = []
for airport in top_airports:
    sample_pd = (
        silver_df
        .where(
            (F.col("origin_airport") == airport)
            & F.col("arrival_delay").isNotNull()
        )
        .select("origin_airport", "arrival_delay")
        .sample(withReplacement=False, fraction=0.2, seed=42)
        .limit(20_000)
        .toPandas()
    )
    groups.append(sample_pd)

df_sample = pd.concat(groups, ignore_index=True)

group_a = df_sample.loc[
    df_sample["origin_airport"] == top_airports[0], "arrival_delay"
].dropna()
group_b = df_sample.loc[
    df_sample["origin_airport"] == top_airports[1], "arrival_delay"
].dropna()

assert len(group_a) >= 2 and len(group_b) >= 2, (
    "Échantillon insuffisant pour le test de Mann-Whitney."
)

stat_1, p_val_1 = stats.mannwhitneyu(
    group_a, group_b, alternative="two-sided"
)

print(f"Test de Mann-Whitney : {top_airports[0]} contre {top_airports[1]}")
print(f"Médianes des retards : {group_a.median():.2f} et {group_b.median():.2f} minutes")
print(f"U = {stat_1:.2f} ; p-value = {p_val_1:.4g}")
print(
    "Conclusion :",
    "les distributions diffèrent significativement (seuil 5 %)."
    if p_val_1 < 0.05
    else "pas de différence significative détectée (seuil 5 %)."
)

# Tester la corrélation entre distance et retard sur un échantillon limité.
spearman_pd = (
    silver_df
    .select("distance", "arrival_delay")
    .where(
        F.col("distance").isNotNull()
        & F.col("arrival_delay").isNotNull()
    )
    .sample(withReplacement=False, fraction=0.05, seed=42)
    .limit(30_000)
    .toPandas()
    .dropna()
)

assert len(spearman_pd) >= 3, "Échantillon insuffisant pour le test de Spearman."

rho, p_val_2 = stats.spearmanr(
    spearman_pd["distance"], spearman_pd["arrival_delay"]
)

print("\nTest de Spearman : distance et retard à l'arrivée")
print(f"rho = {rho:.4f} ; p-value = {p_val_2:.4g}")
print(
    "Conclusion :",
    "association monotone significative (seuil 5 %)."
    if p_val_2 < 0.05
    else "pas d'association monotone significative détectée (seuil 5 %)."
)

26/11/19 10:10:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Test de Mann-Whitney : ATL contre ORD
Médianes des retards : -6.00 et -4.00 minutes
U = 188401925.50 ; p-value = 9.54e-24
Conclusion : les distributions diffèrent significativement (seuil 5 %).


/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()



Test de Spearman : distance et retard à l'arrivée
rho = -0.0665 ; p-value = 9.526e-31
Conclusion : association monotone significative (seuil 5 %).


## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [5]:
SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
GOLD_SEGMENTS_PATH = "/lakehouse/gold/segments_aeroports"

silver_df = spark.read.format("delta").load(SILVER_MAIN_PATH)

# Résumer les vols par aéroport côté Spark, avant de transférer ce petit tableau
# d'aéroports vers Pandas.
entities_sdf = (
    silver_df
    .groupBy("origin_airport")
    .agg(
        F.count("*").alias("nb_vols"),
        F.percentile_approx(
            F.greatest(F.col("arrival_delay"), F.lit(0)), 0.5
        ).alias("retard_positif_mediane_min"),
        F.avg(
            F.when(
                F.col("arrival_delay").isNotNull(),
                (F.col("arrival_delay") >= 15).cast("double")
            )
        ).alias("taux_retard_15"),
        F.avg(
            F.coalesce(F.col("is_cancelled").cast("double"), F.lit(0.0))
        ).alias("taux_annulation"),
        F.percentile_approx("distance", 0.5).alias("distance_mediane")
    )
)

entities_pd = entities_sdf.toPandas().dropna().copy()

feature_cols = [
    "nb_vols",
    "retard_positif_mediane_min",
    "taux_retard_15",
    "taux_annulation",
    "distance_mediane",
]

assert len(entities_pd) >= 3, "Il faut au moins trois aéroports pour K-Means."

# Les volumes, retards et distances étant positifs et asymétriques,
# log1p réduit l'influence des grandes valeurs.
features = entities_pd[feature_cols].copy()
for col in ["nb_vols", "retard_positif_mediane_min", "distance_mediane"]:
    features[col] = np.log1p(features[col].clip(lower=0))

X = StandardScaler().fit_transform(features)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
entities_pd["cluster_id"] = kmeans.fit_predict(X).astype(int)

# Examiner les profils pour interpréter les groupes.
profile = entities_pd.groupby("cluster_id")[feature_cols].median()
display(profile)

# Nommer les clusters du risque le plus faible au plus élevé.
cluster_order = profile.sort_values(
    ["taux_retard_15", "retard_positif_mediane_min"]
).index.tolist()

labels = ["Faible risque", "Risque intermédiaire", "Risque élevé"]
label_map = {
    cluster_id: labels[i]
    for i, cluster_id in enumerate(cluster_order)
}
entities_pd["segment"] = entities_pd["cluster_id"].map(label_map)

assert "segment" in entities_pd.columns

(
    spark.createDataFrame(entities_pd)
    .coalesce(2)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_SEGMENTS_PATH)
)

print(
    f"Table de segmentation enregistrée dans {GOLD_SEGMENTS_PATH} "
    f"({len(entities_pd):,} aéroports)"
)

/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,nb_vols,retard_positif_mediane_min,taux_retard_15,taux_annulation,distance_mediane
cluster_id,,,,,
0,4423.0,0.0,0.175333,0.019714,516.0
1,280.5,0.0,0.110533,0.004709,310.5
2,86.5,8.5,0.432449,0.006494,1192.0


/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Cannot convert pyarrow.lib.ChunkedArray to pyarrow.lib.Array
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Table de segmentation enregistrée dans /lakehouse/gold/segments_aeroports (628 aéroports)


## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

In [6]:
# Vérification finale de l'inventaire complet de votre couche Gold sur HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")

=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===


 - /lakehouse/gold/retards_mensuels          :        12 lignes


 - /lakehouse/gold/retards_par_compagnie     :        14 lignes
 - /lakehouse/gold/segments_aeroports        :       628 lignes


 - /lakehouse/gold/top_compagnie_retards_par_aeroport :       628 lignes
